In [ ]:
#import squidpy as sq
import scanpy as sc
import scipy as scp
import numpy as np 
import pandas as pd
#import scvi
import seaborn as sns
import pyensembl as ensb
import matplotlib.pyplot as plt
import mygene

In [ ]:
import gseapy as gp
import scanpy as sc

# GSEA

In [ ]:
sclcCells = sc.read_h5ad('data/sclcSingletwGenes.h5ad')

In [ ]:
sclcCells

In [ ]:
sclcCells_normal = sc.read_h5ad('data/normalCellsOld.h5ad') 

In [ ]:
sclcCells_normal.var.index = sclcCells_normal.var.feature_name

In [ ]:
sclcCells_normal.var

In [ ]:
concat = sclcCells.concatenate(sclcCells_1)

In [ ]:
concat

In [ ]:
concat.obs.groupby('cell_type')['disease'].value_counts()

In [ ]:
concat.obs['disease'] = pd.Categorical(concat.obs['disease'], categories=["normal", "squamous cell lung carcinoma"], ordered=True)
indices = concat.obs.sort_values(['cell_type', 'disease']).index
concat = concat[indices,:]

In [ ]:
bdata = concat[concat.obs.cell_type == "type I pneumocyte"].copy()
bdata

In [ ]:
import time
t1 = time.time()
# NOTE: To speed up, use gp.prerank instead with your own ranked list.
res = gp.gsea(data=bdata.to_df().T, # row -> genes, column-> samples
        gene_sets="GO_Biological_Process_2021",
        cls=bdata.obs.disease,
        permutation_num=1000,
        permutation_type='phenotype',
        outdir=None,
        method='s2n', # signal_to_noise
        threads= 16)
t2=time.time()
print(t2-t1)

In [ ]:
res.res2d.head(10)

In [ ]:
## Heatmap of gene expression
i = 7
genes = res.res2d.Lead_genes.iloc[i].split(";")
ax = gp.heatmap(df = res.heatmat.loc[genes],
           z_score=None,
           title=res.res2d.Term.iloc[i],
           figsize=(6,5),
           cmap=plt.cm.viridis,
           xticklabels=False)

In [ ]:
term = res.res2d.Term
# gp.gseaplot(res.ranking, term=term[i], **res.results[term[i]])
axs = res.plot(terms=term[:5])

In [ ]:
# find degs
sc.tl.rank_genes_groups(bdata,
                        groupby='disease',
                        use_raw=False,
                        #layer='lognorm',
                        method='wilcoxon',
                        groups=["squamous cell lung carcinoma"],
                        reference='normal')

In [ ]:
bdata.X.max() # already log1p

In [ ]:
sc.pl.rank_genes_groups(bdata, n_genes=25, sharey=False)

In [ ]:
# get deg result
result = bdata.uns['rank_genes_groups']
groups = result['names'].dtype.names
degs = pd.DataFrame(
    {group.split(' ')[0] + '_' + key: result[key][group]
    for group in groups for key in ['names','scores', 'pvals','pvals_adj','logfoldchanges']})

In [ ]:
degs

In [ ]:
# subset up or down regulated genes
degs_sig = degs[degs.squamous_pvals_adj < 0.05]
degs_up = degs_sig[degs_sig.squamous_logfoldchanges > 0]
degs_dw = degs_sig[degs_sig.squamous_logfoldchanges < 0]

In [ ]:
enr_up = gp.enrichr(degs_up.squamous_names,
                    gene_sets='GO_Biological_Process_2021',
                    outdir=None)

In [ ]:
enr_up.res2d.Term = enr_up.res2d.Term.str.split(" \(GO").str[0]

In [ ]:
enr_dw = gp.enrichr(degs_dw.squamous_names,
                    gene_sets='GO_Biological_Process_2021',
                    outdir=None)

In [ ]:
# concat results
enr_up.res2d['UP_DW'] = "UP"
enr_dw.res2d['UP_DW'] = "DOWN"
enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

In [ ]:
ax = gp.barplot(enr_res, figsize=(3,5),
                group ='UP_DW',
                title ="GO_BP",
                color = ['b','r'])

In [ ]:
import networkx as nx

In [ ]:
res.res2d.head()

In [ ]:
nodes, edges = gp.enrichment_map(res.res2d)

In [ ]:
nodes.head()

In [ ]:
# build graph
G = nx.from_pandas_edgelist(edges,
                            source='src_idx',
                            target='targ_idx',
                            edge_attr=['jaccard_coef', 'overlap_coef', 'overlap_genes'])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

# init node cooridnates
pos=nx.layout.spiral_layout(G)
#node_size = nx.get_node_attributes()
# draw node
nx.draw_networkx_nodes(G,
                       pos=pos,
                       cmap=plt.cm.RdYlBu,
                       node_color=list(nodes.NES),
                       node_size=list(nodes.Hits_ratio *1000))
# draw node label
nx.draw_networkx_labels(G,
                        pos=pos,
                        labels=nodes.Term.to_dict())
# draw edge
edge_weight = nx.get_edge_attributes(G, 'jaccard_coef').values()
nx.draw_networkx_edges(G,
                       pos=pos,
                       width=list(map(lambda x: x*10, edge_weight)),
                       edge_color='#CDDBD4')
plt.show()